## Website Summarizer using Grok (xAI)
Using Grok's OpenAI-compatible API — similar to the day1 summarizer but powered by xAI's Grok model.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
api_key = os.getenv("GROK_API_KEY")

if not api_key:
    print("No Grok API key found — set GROK_API_KEY in your .env file")
else:
    print("Grok API key found")

### Set up the Grok client
Grok exposes an OpenAI-compatible endpoint, so we can reuse the `openai` package — just like we did with Gemini in day2.

In [ ]:
from openai import OpenAI

grok = OpenAI(
    base_url="https://api.x.ai/v1",
    api_key=api_key
)

### Quick test

In [ ]:
response = grok.chat.completions.create(
    model="grok-3",
    messages=[{"role": "user", "content": "Tell me a fun fact about the universe."}]
)

response.choices[0].message.content

### Website Summarizer

In [ ]:
import requests
from bs4 import BeautifulSoup

def fetch_website_contents(url):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        return soup.get_text(separator="\n", strip=True)
    except requests.exceptions.RequestException as e:
        return f"Error fetching website: {e}"

In [ ]:
system_prompt = """
You are an assistant that analyzes the contents of a website
and provides a short summary, ignoring text that might redirect to other pages.
Respond in markdown."""

user_prompt = """Here is a website.
Provide a short summary of this website in markdown.
If it includes news or announcements, summarize them as well.\n\n"""

def message_for(website_text):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt + website_text}
    ]

In [ ]:
def summarize(url):
    website_text = fetch_website_contents(url)
    response = grok.chat.completions.create(
        model="grok-3",
        messages=message_for(website_text)
    )
    return response.choices[0].message.content

In [ ]:
from IPython.display import display, Markdown

display(Markdown(summarize("https://www.tcs.com")))